# 🎯 Lesson 3.5 — Overfitting, Underfitting, and Generalisation

**AI Crash Course for Absolute Beginners**

A model that memorises training data instead of learning patterns fails in the real world. In this notebook:
- **See** overfitting and underfitting visually
- Use **validation curves** to diagnose the bias-variance trade-off
- Apply **regularisation** to fix overfitting
- Use **cross-validation** for reliable estimates
- Add **early stopping** in a neural-network-style loop

> Run each cell with **Shift + Enter**.

In [ ]:
!pip install numpy matplotlib scikit-learn --quiet

import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures   # turns x into [x, x², x³, ...] for polynomial models
from sklearn.linear_model import LinearRegression, Ridge, Lasso   # regression models (Ridge/Lasso add regularisation)
from sklearn.pipeline import Pipeline   # chains preprocessing + model into one reusable object
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     learning_curve, validation_curve)
# cross_val_score  — splits data into K folds, trains on K-1, tests on 1, repeats K times
# learning_curve   — shows how accuracy changes as we add more training data
# validation_curve — shows how accuracy changes as we change one hyperparameter
from sklearn.tree import DecisionTreeClassifier   # a tree-based model prone to overfitting at large depths
from sklearn.datasets import make_classification  # generates a synthetic labelled dataset
from sklearn.metrics import accuracy_score        # fraction of correct predictions

---
## Part 1 — Seeing Overfitting and Underfitting

In [ ]:
# True function: a sine wave
np.random.seed(42)
X_all = np.linspace(0, 2*np.pi, 100)   # 100 evenly spaced x values from 0 to 2π
y_true = np.sin(X_all)                  # the real pattern we want to learn

# Small noisy sample (what we get to train on)
n_train = 15
# np.random.choice picks 15 random indices from 0-99 (without replacement)
idx = np.sort(np.random.choice(100, n_train, replace=False))
# reshape(-1, 1) turns a 1D array like [1,2,3] into a column: [[1],[2],[3]]
# scikit-learn requires X to be 2D (each row = one sample, each column = one feature)
X_train = X_all[idx].reshape(-1, 1)
y_train = y_true[idx] + np.random.normal(0, 0.2, n_train)   # true values + small random noise

degrees = [1, 4, 15]   # polynomial degrees to test: underfitting, good, overfitting
titles  = ["Underfitting\n(degree 1 — too simple)",
           "Good fit\n(degree 4)",
           "Overfitting\n(degree 15 — too complex)"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# zip(axes, degrees, titles) pairs each subplot with a degree and title
for ax, deg, title in zip(axes, degrees, titles):
    # Pipeline chains two steps: first create polynomial features, then fit a linear model
    # ("poly", PolynomialFeatures(deg)) gives the step a name so we can refer to it later
    pipe = Pipeline([("poly", PolynomialFeatures(deg)),
                     ("lr",   LinearRegression())])
    pipe.fit(X_train, y_train)
    # X_all.reshape(-1, 1) converts the full range of x values into 2D for prediction
    y_pred = pipe.predict(X_all.reshape(-1, 1))

    ax.plot(X_all, y_true, "k--", linewidth=1.5, label="True function")
    ax.scatter(X_train, y_train, color="#1a6bc8", zorder=5, label="Training data")
    ax.plot(X_all, y_pred, color="#c8401a", linewidth=2, label=f"Degree {deg}")
    ax.set_ylim(-2.5, 2.5)
    ax.set_title(title)
    ax.legend(fontsize=7)

plt.suptitle("Bias-Variance Trade-off", fontsize=13)
plt.tight_layout()
plt.show()

---
## Part 2 — Train vs Validation Error

In [ ]:
X_clf, y_clf = make_classification(n_samples=500, n_features=20, n_informative=10,
                                   random_state=42)
X_tr, X_val, y_tr, y_val = train_test_split(X_clf, y_clf, test_size=0.3, random_state=42)

depths     = range(1, 25)   # test tree depths from 1 (very simple) to 24 (very complex)
train_accs = []
val_accs   = []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_tr, y_tr)
    train_accs.append(accuracy_score(y_tr, dt.predict(X_tr)))    # accuracy on training data
    val_accs.append(accuracy_score(y_val, dt.predict(X_val)))    # accuracy on unseen validation data

plt.figure(figsize=(8, 4))
plt.plot(depths, train_accs, label="Training accuracy",   color="#1a6bc8", marker="o", ms=4)
plt.plot(depths, val_accs,   label="Validation accuracy", color="#c8401a", marker="s", ms=4)

# np.argmax(val_accs) returns the index of the highest validation accuracy
# +1 because our depth range starts at 1 (not 0), so index 0 = depth 1
plt.axvline(np.argmax(val_accs)+1, color="grey", linestyle="--",
            label=f"Best depth = {np.argmax(val_accs)+1}")
plt.xlabel("Tree Max Depth")
plt.ylabel("Accuracy")
plt.title("Overfitting Becomes Visible When Train >> Validation")
plt.legend()
plt.tight_layout()
plt.show()

---
## Part 3 — Regularisation: The Cure for Overfitting

In [ ]:
# Polynomial degree-10 with and without Ridge regularisation
X_tr10 = X_train  # reuse sine data
y_tr10 = y_train
X_plot = X_all.reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

configs = [
    (LinearRegression(), "No regularisation\n(overfits badly)"),
    (Ridge(alpha=1),     "Ridge alpha=1\n(mild regularisation)"),
    (Ridge(alpha=100),   "Ridge alpha=100\n(strong — near linear)"),
]

for ax, (model, title) in zip(axes, configs):
    pipe = Pipeline([("poly", PolynomialFeatures(10)), ("reg", model)])
    pipe.fit(X_tr10, y_tr10)
    y_pred = pipe.predict(X_plot)

    ax.plot(X_all, np.sin(X_all), "k--", linewidth=1.5, label="True function")
    ax.scatter(X_train.flatten(), y_train, color="#1a6bc8", zorder=5)
    ax.plot(X_all, y_pred, color="#c8401a", linewidth=2)
    ax.set_ylim(-3, 3)
    ax.set_title(title)

plt.suptitle("Ridge Regularisation Controls Overfitting", fontsize=13)
plt.tight_layout()
plt.show()

---
## Part 4 — Cross-Validation: Reliable Evaluation

In [ ]:
# 5-fold cross-validation
# cross_val_score splits data into 5 equal parts (folds)
# It trains on 4 folds and tests on the 1 remaining fold — rotating through all 5
# This gives 5 accuracy scores; averaging them is more reliable than a single train/test split
from sklearn.ensemble import RandomForestClassifier

for model, name in [
    (DecisionTreeClassifier(max_depth=5, random_state=42), "Decision Tree (depth 5)"),
    (RandomForestClassifier(n_estimators=100, random_state=42), "Random Forest (100 trees)")
]:
    # cv=5 means 5-fold cross-validation
    # scoring="accuracy" specifies what metric to measure in each fold
    # n_jobs=-1 uses all available CPU cores to run folds in parallel (faster)
    scores = cross_val_score(model, X_clf, y_clf, cv=5, scoring="accuracy")
    print(f"{name}")
    print(f"  Fold scores : {[f'{s:.3f}' for s in scores]}")
    print(f"  Mean +/- std  : {scores.mean():.3f} +/- {scores.std():.3f}\n")

---
## Part 5 — Learning Curves: Diagnose with More Data

In [ ]:
dt = DecisionTreeClassifier(max_depth=5, random_state=42)

# learning_curve trains the model on increasing amounts of data and measures accuracy
# train_sizes=np.linspace(0.1, 1.0, 10) means 10 steps from 10% of data up to 100%
# cv=5 means each size is evaluated using 5-fold cross-validation
# n_jobs=-1 runs the folds in parallel using all CPU cores
train_sizes, train_scores, val_scores = learning_curve(
    dt, X_clf, y_clf, cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10)
)

# train_scores is a 2D array: rows = training sizes, columns = CV fold scores
# .mean(axis=1) averages across the 5 folds for each training size
train_mean = train_scores.mean(axis=1)
val_mean   = val_scores.mean(axis=1)

plt.figure(figsize=(8, 4))
plt.plot(train_sizes, train_mean, label="Train", color="#1a6bc8", marker="o")
plt.plot(train_sizes, val_mean,   label="Validation", color="#c8401a", marker="s")

# fill_between adds a shaded band showing the standard deviation (how spread out the fold scores are)
# A wide band means the model is inconsistent — narrow band means stable performance
plt.fill_between(train_sizes,
                 train_mean - train_scores.std(axis=1),
                 train_mean + train_scores.std(axis=1), alpha=0.15, color="#1a6bc8")
plt.fill_between(train_sizes,
                 val_mean   - val_scores.std(axis=1),
                 val_mean   + val_scores.std(axis=1), alpha=0.15, color="#c8401a")
plt.xlabel("Training Set Size")
plt.ylabel("Accuracy")
plt.title("Learning Curves — if gap is large, model overfits")
plt.legend()
plt.tight_layout()
plt.show()

---
## Summary Checklist

| Symptom | Diagnosis | Fix |
|---|---|---|
| Train good, val bad | Overfitting | Regularise, more data, simpler model |
| Both bad | Underfitting | More complex model, more features |
| Inconsistent val scores | High variance | Cross-validation, ensemble methods |
| Val improves with more data | Data-limited | Collect more training data |

---
## Challenge Exercises

1. **Dropout simulation**: In the sine wave example, randomly zero out 30% of training samples on each fit call (`np.random.choice`) — how does the model behave?
2. **Lasso vs Ridge**: Fit both with `alpha=1` on the sine data (degree 8) — which coefficients does Lasso zero out?
3. **Early stopping**: Simulate a training loop where you track val loss each epoch and stop when it hasn't improved for 5 epochs.

---
**Next lesson:** [Lesson 3.6 — Model Evaluation](https://colab.research.google.com/github/GirlEf/ai-crash-course/blob/main/notebooks/lesson-3.6-evaluation.ipynb)